In [1]:
import pandas as pd
import mlflow
import os
from dotenv import load_dotenv
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import confusion_matrix

/home/vladislave/Desktop/AAA_MLSD/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# загружаем переменные окружения и объявляем константы 

load_dotenv()

MFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
LABELED_DATA_PATH = os.getenv("LABELED_DATA_PATH")

RANDOM_SEED = 42

In [4]:
data_v1 = pd.read_csv(LABELED_DATA_PATH).dropna()
data_v1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 369360 entries, 0 to 369368
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   text        369360 non-null  object
 1   item_title  369360 non-null  object
 2   sentiment   369360 non-null  object
 3   synthetic   369360 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 14.1+ MB


In [5]:
data_v1[["sentiment", "synthetic"]].value_counts()

sentiment   synthetic
normal      0            356954
external    0              4327
            1              3675
spam        1              1878
harassment  1              1653
threat      1               562
spam        0               220
harassment  0                87
threat      0                 4
Name: count, dtype: int64

# Baseline model 

Для начала соберем тестовый и тренировочный датасет следующим образом :

1. В `VAL_DATA` попадут 8_000 сообщений класса `normal` и по половине из всех несинтетических сообщений остальных классов
2. Для класса `threat` добавим 100 синтетических примеров (то есть из другого датасета), считая их подходящими для домена

Итого обучаться мы будем на синтетике и половине примеров из реального датасета, а оцениваться - только на реальных примерах (за исключением класса `threat`, который предвставлен всего в 2 примерах,
поэтому для него добавим в валидацию примеров из других датасетов)

In [15]:
synthetic_data = data_v1[data_v1["synthetic"] == 1]
base_data = data_v1[data_v1["synthetic"] == 0]

real_not_normal = base_data[base_data["sentiment"] != "normal"]
val_real_half = real_not_normal.groupby("sentiment", group_keys=False).apply(lambda x: x.sample(frac=0.5, random_state=RANDOM_SEED))
train_real_half = real_not_normal.drop(index=val_real_half.index)

# берем 8000 сообщений класса normal и в валидацию, и в трейн (без пересечений)
real_normal = base_data[base_data["sentiment"] == "normal"]
normal_sampled = real_normal.sample(n=8_000, random_state=RANDOM_SEED)
normal_8000 = real_normal.drop(index=normal_sampled.index, errors="ignore").sample(n=8_000, random_state=RANDOM_SEED)

# в трейн кидаем синтетику
train_harassment = synthetic_data[synthetic_data["sentiment"] == "harassment"]
train_threat = synthetic_data[synthetic_data["sentiment"] == "threat"]
train_spam = synthetic_data[synthetic_data["sentiment"] == "spam"]
train_external = synthetic_data[synthetic_data["sentiment"] == "external"]

VAL_DATA = pd.concat([val_real_half, normal_sampled])
TRAIN_DATA = pd.concat([normal_8000,train_harassment,train_threat,train_spam,train_external,train_real_half])

# добавляем еще 100 примеров из других датасетов, чтобы метрики не были шумными
val_addition_threat_sampled = train_threat.sample(n=100, random_state=RANDOM_SEED)

VAL_DATA = pd.concat([VAL_DATA, val_addition_threat_sampled])
TRAIN_DATA = TRAIN_DATA.drop(index=val_addition_threat_sampled.index)

/tmp/ipykernel_1515056/1957064016.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  val_real_half = real_not_normal.groupby("sentiment", group_keys=False).apply(lambda x: x.sample(frac=0.5, random_state=RANDOM_SEED))


In [237]:
VAL_DATA[['sentiment', 'synthetic']].value_counts()

sentiment   synthetic
normal      0            8000
external    0            2164
spam        0             110
threat      1             100
harassment  0              44
threat      0               1
Name: count, dtype: int64

In [11]:
TRAIN_DATA[['sentiment', 'synthetic']].value_counts()

sentiment   synthetic
normal      0            8000
external    1            3675
harassment  1            2626
spam        1            2488
external    0            2163
threat      1             462
spam        0             110
harassment  0              43
threat      0               2
Name: count, dtype: int64

Теперь переменная `VAL_DATA` содержит валидационный датасет, состоящий из реальных примером (за исключением класса `threat`, который из-за своей маленькости был расширен).
В переменной `TRAIN_DATA` содержатся все синтетические данные и по половине сообщений для классов, отличных от `normal`

In [12]:
# обучим самую простую модель на этих данных и посмотрим на classification report

X_train, y_train = TRAIN_DATA['text'], TRAIN_DATA['sentiment']
X_test, y_test = VAL_DATA['text'], VAL_DATA['sentiment']

pipeline = Pipeline(
    [
        ("tfidf", TfidfVectorizer()),
        ("clf", LogisticRegression(random_state=RANDOM_SEED)),
    ]
)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    external       0.77      0.75      0.76      2164
  harassment       0.25      0.70      0.37        44
      normal       0.93      0.93      0.93      8000
        spam       0.41      0.54      0.46       110
      threat       0.80      0.40      0.54       102

    accuracy                           0.88     10420
   macro avg       0.63      0.66      0.61     10420
weighted avg       0.89      0.88      0.88     10420



# Improve baseline model

В этой части ноубука будет проведена серия экспериментов с архитектурами и параметрами. Среди них будет:
1. Подбор параметров для tf-idf + logistic regression
2. Эмбединги на основе предобученной bert модели + logistic regression
3. Эмбединги на основе предобученной bert модели + catboost
4. Дообучение классификатора на основе предобученной bert модели

Все эксперименты будут логироваться в *MLFlow* для удобного сравнения и анализа в будущем


In [ ]:
# подключаемся к серверу mlflow, где будут храниться метрики и информация об обучении модели

mlflow.set_tracking_uri(MFLOW_TRACKING_URI)
mlflow.set_experiment(experiment_id="1")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1780847527484, experiment_id='1', last_update_time=1780847527484, lifecycle_stage='active', name='tfidf_logreg', tags={}, trace_location=None, workspace='default'>

In [5]:
def log_mlflow(model: Pipeline, model_type: str, metrics: dict, params: dict, artifacts: list[str], run_name: str) -> None:
    """
    Функция, логирующая параметры, метрики и артифакты в mlflow
    """
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        if artifacts:
            for artifact in artifacts:
                mlflow.log_artifact(artifact)

        if model_type == "sklearn":
            mlflow.sklearn.log_model(model, "model")

        elif model_type == "catboost":
            mlflow.catboost.log_model(model, "model")

In [ ]:
def train_tfidf_logreg(X_train: pd.DataFrame, y_train: pd.Series,tf_params: dict, lr_params:dict) -> Pipeline:
    """
    Обучает и возвращает объект Pipeline. Текстовые поля преобразуются с помощью TfidfVectorizer
    На полученных эмбедингах обучается LogisticRegression
    """
    pipeline = Pipeline(
        [
            ("tfidf", TfidfVectorizer(**tf_params) if tf_params else TfidfVectorizer()),
            ("clf", LogisticRegression(**lr_params)),
        ]
    )

    pipeline.fit(X_train, y_train)
    return pipeline


def save_confusion_matrix_png(y_true: pd.Series, y_pred: np.ndarray, path_png: str) -> str:
    """
    Строит матрицу ошибок, визуализирует её сохраняет в PNG и возвращает путь к файлу.
    """

    dir_name = os.path.dirname(path_png)
    if dir_name:
        os.makedirs(dir_name, exist_ok=True)
        
    classes = sorted(list(set(np.unique(y_true)) | set(np.unique(y_pred))))
    cm = confusion_matrix(y_true, y_pred, labels=classes)

    # Матрицу будут сохранять в .png при помощи функции heatmap из seaborn
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d",cmap="Blues",xticklabels=classes,yticklabels=classes,square=True)
    plt.title("Confusion Matrix", fontsize=16, pad=20, fontweight='bold')
    plt.ylabel("Actual Labels", fontsize=12, labelpad=10)
    plt.xlabel("Predicted Labels", fontsize=12, labelpad=10)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(path_png, dpi=300)
    plt.close()
    
    return path_png


def get_metrics(y_test: pd.Series, y_pred: pd.Series) -> dict:
    """
    Возвращает словарь с основными метриками, среди которых precision,
    recall, f1, support для каждого класса в разрезе.
    """
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    class_metrics = {
        cls: {
            "precision": report[cls]["precision"],
            "recall": report[cls]["recall"],
            "f1": report[cls]["f1-score"],
            "support": report[cls]["support"],
        }
        for cls in report
        if cls not in ["accuracy", "macro avg", "weighted avg"]
    }

    metrics = {}
    for cls, vals in class_metrics.items():
        metrics[f"{cls}_precision"] = vals["precision"]
        metrics[f"{cls}_recall"] = vals["recall"]
        metrics[f"{cls}_f1"] = vals["f1"]
        metrics[f"{cls}_support"] = vals["support"]

    return metrics

In [8]:
# tfidf параметры
max_features = [1000, None]
ngram_range = [(1, 1), (1, 2), (1, 3)]
min_df = [3, 5]

# lr параметры
C = [0.1, 1.0, 5.0, 10.0, 7.5, 0.5]
solver = ["saga", "lbfgs"]
max_iter = [100, 500, 1000]
class_weight = ["balanced"]

# формируем сетку параметров
param_grid = {
    "max_features": max_features,
    "ngram_range": ngram_range,
    "min_df": min_df,
    "C": C,
    "solver": solver,
    "max_iter": max_iter,
    "class_weight": class_weight,
}

In [ ]:
# перебираем все возможные комбинации параметров

for idx, params in enumerate(ParameterGrid(param_grid)):
    tf_params = {
        "max_features": params["max_features"],
        "ngram_range": params["ngram_range"],
        "min_df": params["min_df"],
    }

    lr_params = {
        "C": params["C"],
        "solver": params["solver"],
        "max_iter": params["max_iter"],
        "class_weight": params["class_weight"],
    }

    # train-test-split
    X_train, y_train = TRAIN_DATA['text'], TRAIN_DATA['sentiment']
    X_test, y_test = VAL_DATA['text'], VAL_DATA['sentiment']

    # обучаем tfidf + logreg и сохраняем метрики
    model = train_tfidf_logreg(X_train, y_train, tf_params, lr_params)

    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)

    os.makedirs("./temp_cms", exist_ok=True)
    train_path_png = f"./temp_cms/train_cm_run_{idx}.png"
    test_path_png = f"./temp_cms/test_cm_run_{idx}.png"

    # пути до матриц ошибок на трейне и тесте
    path_train_cm = save_confusion_matrix_png(y_true=y_train, y_pred=train_preds, path_png=train_path_png)
    path_test_cm = save_confusion_matrix_png(y_true=y_test, y_pred=test_preds, path_png=test_path_png)

    metrics = get_metrics(y_test, pd.Series(test_preds))
    
    log_mlflow(
        model=model,
        model_type="sklearn",
        metrics=metrics,
        params=tf_params | lr_params,
        artifacts=[path_train_cm, path_test_cm], # матрицы ошибок будут в артефактах модели
        run_name=f"logreg_tfidf_run_{idx}"
    )

Результаты записаны на mlflow и будут участвовать в сравнении далее

# BERT + Logreg (Catboost) or STF

В этом блоке в качестве слоя-эмбедингов будут выступать разные предобученные bert модели. Попробуем и залогируем как логистические регрессии, обученные на эмбедингах, так и дообученные bert модели.

Первой будет следующая модель: https://huggingface.co/cointegrated/rubert-tiny-sentiment-balanced. Из ее описания на HF можно найти, что она обучалась на коротких текстах на русском языке для задачи классификации: 

*This is the cointegrated/rubert-tiny model fine-tuned for classification of sentiment for short Russian texts. \
The problem is formulated as multiclass classification: negative vs neutral vs positive*

In [7]:
from sentence_transformers import SentenceTransformer
from transformers import set_seed
from catboost import CatBoostClassifier

set_seed(RANDOM_SEED)

In [52]:
model = SentenceTransformer("cointegrated/rubert-tiny-sentiment-balanced")

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 2632.03it/s]
[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny-sentiment-balanced
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
def train_bert_logreg(X_train_embeddings: np.ndarray, y_train: pd.Series, lr_params: dict = None) -> LogisticRegression:
    logreg = LogisticRegression(**lr_params)
    logreg.fit(X_train_embeddings, y_train)
    return logreg

def train_bert_catboost(X_train_embeddings: np.ndarray, y_train: pd.Series, cat_params: dict = None) -> CatBoostClassifier:
    catboost = CatBoostClassifier(verbose=False, task_type='GPU', **cat_params)
    catboost.fit(X_train_embeddings, y_train)
    return catboost

## Logreg + bert

Тут пробуем на эмбедингах от bert обучить логистическую регрессию

In [ ]:
mlflow.set_experiment(experiment_id="2")

In [ ]:
# формируем сетку параметров (все те же что и для логрега, но без tfidf)
C = [0.1, 1.0, 10.0, 0.5]
solver = ["saga", "lbfgs"]
max_iter = [1000, 3000]
class_weight = ["balanced"]

lr_grid = {
    "C": C,
    "solver": solver,
    "max_iter": max_iter,
    "class_weight": class_weight,
}

In [ ]:
# тот же цикл обучения, но без параметров для tf-idf. сохраняем те же метрики и артефакты
# сразу сделаем эмбединги из текстовых колонок и будем обучать

X_train, y_train = TRAIN_DATA['text'], TRAIN_DATA['sentiment']
X_test, y_test = VAL_DATA['text'], VAL_DATA['sentiment']

X_train_embeddings = model.encode(X_train.to_list())
X_test_embeddings = model.encode(X_test.to_list())

# перебираем все возможные сочетания параметров
for idx, params in enumerate(ParameterGrid(lr_grid)):
    lr_params = {
        "C": params["C"],
        "solver": params["solver"],
        "max_iter": params["max_iter"],
        "class_weight": params["class_weight"],
    }

    model_logreg_bert = train_bert_logreg(X_train_embeddings, y_train, lr_params)

    train_preds = model_logreg_bert.predict(X_train_embeddings)
    test_preds = model_logreg_bert.predict(X_test_embeddings)

    os.makedirs("./temp_cms", exist_ok=True)
    train_path_png = f"./temp_cms/train_cm_run_{idx}.png"
    test_path_png = f"./temp_cms/test_cm_run_{idx}.png"
    path_train_cm = save_confusion_matrix_png(y_true=y_train, y_pred=train_preds, path_png=train_path_png)
    path_test_cm = save_confusion_matrix_png(y_true=y_test, y_pred=test_preds, path_png=test_path_png)

    metrics = get_metrics(y_test, pd.Series(test_preds))
    
    log_mlflow(
        model=model,
        model_type="sklearn",
        metrics=metrics,
        params=lr_params,
        artifacts=[path_train_cm, path_test_cm],
        run_name=f"logreg_bert_run_{idx}"
    )

## Catboost + bert

Тут пробуем на эмбедингах от bert обучить градиентный бустинг (на примере catboost)

In [54]:
mlflow.set_experiment(experiment_id="4")

<Experiment: artifact_location='mlflow-artifacts:/4', creation_time=1780935174121, experiment_id='4', last_update_time=1780935174121, lifecycle_stage='active', name='bert_catboost', tags={}, trace_location=None, workspace='default'>

In [60]:
# Формируем сетку параметров для многоклассового CatBoost
iterations = [500, 1000]
learning_rate = [0.03, 0.1, 0.2]
depth = [4, 6, 8]
l2_leaf_reg = [1, 3, 5, 10]
loss_function = ["MultiClass"]
auto_class_weights = ["Balanced"]

cb_grid = {
    "iterations": iterations,
    "learning_rate": learning_rate,
    "depth": depth,
    "l2_leaf_reg": l2_leaf_reg,
    "loss_function": loss_function,
    "auto_class_weights": auto_class_weights,
}

In [ ]:
# аналогичный блок обучения, что и был для логистической регрессии

X_train, y_train = TRAIN_DATA['text'], TRAIN_DATA['sentiment']
X_test, y_test = VAL_DATA['text'], VAL_DATA['sentiment']

X_train_embeddings = model.encode(X_train.to_list())
X_test_embeddings = model.encode(X_test.to_list())

for idx, params in enumerate(ParameterGrid(cb_grid)):
    model_catboost_bert = train_bert_catboost(X_train_embeddings, y_train, params)

    train_preds = model_catboost_bert.predict(X_train_embeddings)
    test_preds = model_catboost_bert.predict(X_test_embeddings)
    
    train_preds_flat = train_preds.flatten()
    test_preds_flat = test_preds.flatten()

    os.makedirs("./temp_cms", exist_ok=True)
    train_path_png = f"./temp_cms/train_cm_run_{idx}.png"
    test_path_png = f"./temp_cms/test_cm_run_{idx}.png"
    
    path_train_cm = save_confusion_matrix_png(y_true=y_train, y_pred=train_preds_flat, path_png=train_path_png)
    path_test_cm = save_confusion_matrix_png(y_true=y_test, y_pred=test_preds_flat, path_png=test_path_png)

    metrics = get_metrics(y_test, pd.Series(test_preds_flat))
    
    log_mlflow(
        model=model_catboost_bert,
        model_type="catboost",
        metrics=metrics,
        params=params,
        artifacts=[path_train_cm, path_test_cm],
        run_name=f"catboost_bert_run_{idx}"
    )

## BERT classifier 

In [9]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight

In [10]:
def log_mlflow_bert(path_to_bert_model, metrics, artifacts, run_name):
    model = AutoModelForSequenceClassification.from_pretrained(path_to_bert_model)
    tokenizer = AutoTokenizer.from_pretrained(path_to_bert_model)
    
    with mlflow.start_run(run_name=run_name):
        mlflow.log_metrics(metrics)
        if artifacts:
            for artifact in artifacts:
                mlflow.log_artifact(artifact)
        
        mlflow.transformers.log_model(
            transformers_model={"model": model, "tokenizer": tokenizer},
            artifact_path="bert_model",
            task="text-classification"
        )

### rubert-tiny-sentiment-balanced

В начале попробуем дообучить тот же самый bert, который был использован ранее для получения эмбедингов

In [11]:
model_name = "cointegrated/rubert-tiny-sentiment-balanced"
number_of_labels = 5 # normal, external, spam, harassment, threat

label_to_number = {label: i for i, label in enumerate(data_v1["sentiment"].unique())}
number_to_label = {i: label for label, i in label_to_number.items()}

In [12]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=number_of_labels,
    id2label=number_to_label,
    label2id=label_to_number,
    ignore_mismatched_sizes=True
)

[transformers] You passed `num_labels=5` which is incompatible to the `id2label` map of length `3`.
Loading weights: 100%|██████████| 57/57 [00:00<00:00, 3617.80it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny-sentiment-balanced
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([5])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 312]) vs model:torch.Size([5, 312])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [13]:
def tokenize_function(examples: pd.Series) -> list[int]:
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)


def compute_metrics(eval_pred: np.array) -> dict:
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    y_test_series = pd.Series(labels)
    y_pred_series = pd.Series(predictions)
    calculated_metrics = get_metrics(y_test_series, y_pred_series)

    # сумма f1 скоров нужна лишь для того, чтобы при валидации trainer знал, как выбрать лучшую модель
    # чтобы код работал корректно. ориентироваться мы будем на остальные метрики отдельно, сохраняться модель будет каждую эпоху
    f1_values = [value for key, value in calculated_metrics.items() if key.endswith('_f1')]
    calculated_metrics["sum_f1"] = sum(f1_values)
    
    return calculated_metrics

In [14]:
X_train, y_train = TRAIN_DATA["text"], TRAIN_DATA["sentiment"].map(label_to_number) # мапим классы на цифры

eval_dataset = Dataset.from_pandas(pd.DataFrame({"text": VAL_DATA["text"], "label": VAL_DATA["sentiment"].map(label_to_number)}))
train_dataset = Dataset.from_pandas(pd.DataFrame({"text": X_train, "label": y_train}))

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 10420/10420 [00:01<00:00, 6872.20 examples/s]


Для того, чтобы обучение шло лучше, необходимо поработать с дисбалансом классом. Для этого будем использоваться взвешенный лосс.
Нативно объект trainer не имеет такого выбора, поэтому обернем его в класс `WeightedTrainer` с модифицированным лоссом

In [15]:
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(y_train), y=y_train)
device = "cuda"
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Стандартный CrossEntropyLoss из torch может быть модифицирован
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [16]:
training_args = TrainingArguments(
    output_dir="./bert_logreg_model",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    logging_dir="./bert_logs",
    logging_steps=250,
    learning_rate=2e-5,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    seed=RANDOM_SEED,
    data_seed=RANDOM_SEED,
    metric_for_best_model="sum_f1",
    greater_is_better=True,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    compute_metrics=compute_metrics,
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [24]:
number_to_label

{0: 'normal', 1: 'external', 2: 'spam', 3: 'harassment', 4: 'threat'}

In [32]:
trainer.train()

Epoch,Training Loss,Validation Loss,0 Precision,0 Recall,0 F1,0 Support,1 Precision,1 Recall,1 F1,1 Support,2 Precision,2 Recall,2 F1,2 Support,3 Precision,3 Recall,3 F1,3 Support,4 Precision,4 Recall,4 F1,4 Support,Sum F1
1,0.441944,0.288762,0.990071,0.917400,0.952351,5000.000000,0.842688,0.985213,0.908394,2164.000000,0.489362,0.418182,0.450980,110.000000,0.195489,0.590909,0.293785,44.000000,1.000000,0.294118,0.454545,102.000000,3.060056
2,0.340497,0.284477,0.991859,0.926000,0.957799,5000.000000,0.876690,0.988909,0.929425,2164.000000,0.576471,0.445455,0.502564,110.000000,0.200000,0.613636,0.301676,44.000000,0.758242,0.676471,0.715026,102.000000,3.406489
3,0.279530,0.273094,0.989820,0.933400,0.960782,5000.000000,0.880116,0.983826,0.929086,2164.000000,0.520833,0.454545,0.485437,110.000000,0.290698,0.568182,0.384615,44.000000,0.769231,0.784314,0.776699,102.000000,3.536619
4,0.218015,0.328978,0.992223,0.918600,0.953993,5000.000000,0.858059,0.988909,0.918849,2164.000000,0.417323,0.481818,0.447257,110.000000,0.244898,0.545455,0.338028,44.000000,0.944444,0.666667,0.781609,102.000000,3.439737
5,0.188811,0.313745,0.992055,0.924000,0.956819,5000.000000,0.872750,0.985675,0.925781,2164.000000,0.421053,0.509091,0.460905,110.000000,0.242718,0.568182,0.340136,44.000000,0.891566,0.725490,0.800000,102.000000,3.483642


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 19.02it/s]


TrainOutput(global_step=9370, training_loss=0.33811530293370007, metrics={'train_runtime': 1049.5117, 'train_samples_per_second': 71.395, 'train_steps_per_second': 8.928, 'total_flos': 552765472389120.0, 'train_loss': 0.33811530293370007, 'epoch': 5.0})

In [ ]:
# лучшая по всем метрикам модель имеет наибольший Sum_f1, поэтому именно ее сохранит встроенный метод
trainer.save_model("./best_bert_tiny")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.40it/s]


In [ ]:
# Загрузим метрики на трейне и тесте в mlflow 

logits_test = trainer.predict(tokenized_eval_dataset).predictions
logits_train = trainer.predict(tokenized_train_dataset).predictions

predictions_test = np.argmax(logits_test, axis=-1)
predictions_train = np.argmax(logits_train, axis=-1)

y_test_series = pd.Series(VAL_DATA['sentiment'].map(label_to_number))
y_train_series = pd.Series(TRAIN_DATA['sentiment'].map(label_to_number))

y_pred_series = pd.Series(predictions_test)
calculated_metrics_bert = get_metrics(y_test_series, y_pred_series)
path_train_cm_bert = save_confusion_matrix_png(y_true=y_train_series, y_pred=predictions_train, path_png='model/temp_cms')
path_test_cm_bert = save_confusion_matrix_png(y_true=y_test_series, y_pred=predictions_test, path_png='model/temp_cms')

In [ ]:
mlflow.set_experiment(experiment_id="3")
log_mlflow_bert(path_to_bert_model = "./best_bert_tiny", metrics = calculated_metrics_bert, artifacts = [path_train_cm_bert, path_test_cm_bert], run_name = 'tiny_bert_v1')

<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1780904805592, experiment_id='3', last_update_time=1780904805592, lifecycle_stage='active', name='bert_ft', tags={}, trace_location=None, workspace='default'>

### cointegrated/rubert-tiny2-cedr-emotion-detection

Следущей моделью будет эта : https://huggingface.co/cointegrated/rubert-tiny2-cedr-emotion-detection \
Она была обучена для задачи распознования тольности сообщений, а значит может содержать "полезную информацию" и для нашей задачи

In [ ]:
import torch.nn as nn

model_name = 'cointegrated/rubert-tiny2-cedr-emotion-detection'
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=number_of_labels,
    ignore_mismatched_sizes=True,
    problem_type="single_label_classification"
)

head = nn.Sequential(
    nn.Linear(model.config.hidden_size, 512),
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(512, number_of_labels)
)

model.classifier = head

[transformers] You passed `num_labels=5` which is incompatible to the `id2label` map of length `6`.
Loading weights: 100%|██████████| 57/57 [00:00<00:00, 3238.10it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2-cedr-emotion-detection
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([6, 312]) vs model:torch.Size([5, 312])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([6]) vs model:torch.Size([5])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Далее в точности такой же процесс токенизации и обучения (те же параметры)

In [70]:
def tokenize_function(examples: pd.Series) -> list[int]:
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=1024)

In [71]:
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 10420/10420 [00:02<00:00, 4737.76 examples/s]


In [72]:
training_args = TrainingArguments(
    output_dir="./bert_3",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    logging_dir="./bert_logs",
    logging_steps=250,
    learning_rate=2e-5,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    seed=RANDOM_SEED,
    data_seed=RANDOM_SEED,
    metric_for_best_model="sum_f1",
    greater_is_better=True,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    compute_metrics=compute_metrics,
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [18]:
trainer.train()

Epoch,Training Loss,Validation Loss,0 Precision,0 Recall,0 F1,0 Support,1 Precision,1 Recall,1 F1,1 Support,2 Precision,2 Recall,2 F1,2 Support,3 Precision,3 Recall,3 F1,3 Support,4 Precision,4 Recall,4 F1,4 Support,Sum F1
1,0.428494,0.273568,0.990221,0.911375,0.949164,8000.000000,0.794998,0.969501,0.873621,2164.000000,0.320988,0.472727,0.382353,110.000000,0.119403,0.545455,0.195918,44.000000,0.709091,0.382353,0.496815,102.000000,2.897871
2,0.345404,0.282249,0.995285,0.923500,0.958050,8000.000000,0.807663,0.983826,0.887083,2164.000000,0.382166,0.545455,0.449438,110.000000,0.209524,0.500000,0.295302,44.000000,0.767677,0.745098,0.756219,102.000000,3.346092
3,0.270764,0.309910,0.994057,0.920000,0.955596,8000.000000,0.826562,0.977819,0.895851,2164.000000,0.297980,0.536364,0.383117,110.000000,0.156069,0.613636,0.248848,44.000000,0.776471,0.647059,0.705882,102.000000,3.189294
4,0.259633,0.260049,0.987674,0.941500,0.964034,8000.000000,0.860125,0.951941,0.903707,2164.000000,0.369942,0.581818,0.452297,110.000000,0.192593,0.590909,0.290503,44.000000,0.769231,0.686275,0.725389,102.000000,3.335929
5,0.122299,0.268414,0.989612,0.940750,0.964563,8000.000000,0.848497,0.965342,0.903156,2164.000000,0.420290,0.527273,0.467742,110.000000,0.203125,0.590909,0.302326,44.000000,0.816092,0.696078,0.751323,102.000000,3.389109


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  9.69it/s]


TrainOutput(global_step=11245, training_loss=0.323674717008829, metrics={'train_runtime': 1322.5143, 'train_samples_per_second': 67.999, 'train_steps_per_second': 8.503, 'total_flos': 663421846149120.0, 'train_loss': 0.323674717008829, 'epoch': 5.0})

Метрики просели, модель можно не логировать

### cointegrated/rubert-tiny2-cedr-emotion-detection (5000 normal messages)

Здесь изменено количество сообщений класса normal (немного другое распределение) и на тех же параметров обучен тот же bert

In [107]:
trainer.train()

Epoch,Training Loss,Validation Loss,0 Precision,0 Recall,0 F1,0 Support,1 Precision,1 Recall,1 F1,1 Support,2 Precision,2 Recall,2 F1,2 Support,3 Precision,3 Recall,3 F1,3 Support,4 Precision,4 Recall,4 F1,4 Support,Sum F1
1,0.364456,0.287240,0.991319,0.913600,0.950874,5000.000000,0.864689,0.983364,0.920216,2164.000000,0.473684,0.409091,0.439024,110.000000,0.216216,0.727273,0.333333,44.000000,0.685185,0.725490,0.704762,102.000000,3.348210
2,0.284897,0.242284,0.986837,0.944600,0.965256,5000.000000,0.884599,0.974122,0.927205,2164.000000,0.592593,0.436364,0.502618,110.000000,0.333333,0.613636,0.432000,44.000000,0.865169,0.754902,0.806283,102.000000,3.633362
3,0.223097,0.284420,0.992761,0.932600,0.961741,5000.000000,0.876853,0.983826,0.927265,2164.000000,0.519231,0.490909,0.504673,110.000000,0.301075,0.636364,0.408759,44.000000,0.846939,0.813725,0.830000,102.000000,3.632438
4,0.178047,0.312664,0.994643,0.928400,0.960381,5000.000000,0.870507,0.984750,0.924111,2164.000000,0.432836,0.527273,0.475410,110.000000,0.340909,0.681818,0.454545,44.000000,0.939759,0.764706,0.843243,102.000000,3.657690
5,0.130250,0.291417,0.990890,0.935400,0.962346,5000.000000,0.881299,0.977819,0.927054,2164.000000,0.467742,0.527273,0.495726,110.000000,0.357143,0.681818,0.468750,44.000000,0.890110,0.794118,0.839378,102.000000,3.693254


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.35it/s]


TrainOutput(global_step=9370, training_loss=0.2945916950384606, metrics={'train_runtime': 1100.0271, 'train_samples_per_second': 68.117, 'train_steps_per_second': 8.518, 'total_flos': 552765472389120.0, 'train_loss': 0.2945916950384606, 'epoch': 5.0})

Метрики улучшились очень незначительно, но все равно сохраняем для дальнейшего сравнения

In [108]:
trainer.save_model("./best_bert_tiny_2_n5000")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.84it/s]
